# VF-1 Phase 3A.1 — S1-mini LoRA Smoke Test (Portable Colab Runbook)

**Status: PREPARED & PORTABLE.** This notebook is self-contained and reproducible from a fresh GitHub clone.

### Strict Safety & Approval Gates
The trainer **hard-refuses** to start training unless `gates.json` records four verified approvals:
1. `data_verified`: Dataset integrity and split isolation signed off.
2. `human_audit_approved`: Real human auditor completed spot-check audit.
3. `permissions_cleared`: Written data training permission from Hark or permitted clean data.
4. `user_gpu_approval`: User explicit GPU run authorization.

The base checkpoint `superwhisper/s1-mini` is never modified; only the PEFT LoRA adapter is saved.

In [ ]:
# 1. Environment & GPU Hardware Check
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# 2. Install Pinned Dependencies (keeping preinstalled CUDA PyTorch)
!pip -q install "transformers==5.19.0" "peft==0.21.2" "accelerate==1.15.0" "tokenizers==0.23.3" jinja2 jsonschema pytest pandas

In [ ]:
# 3. Load Clean Engineering Source Bundle & Fetch Tokenizers
# Unpacks s1_smoke_colab_bundle.zip (verified SHA-256; zero restricted Hark data)
import os, zipfile

BUNDLE_ZIP = "/content/s1_smoke_colab_bundle.zip"
APF_DIR = "/content/apf"
os.makedirs(APF_DIR, exist_ok=True)

if os.path.exists(BUNDLE_ZIP):
    print(f"[*] Unpacking {BUNDLE_ZIP} into {APF_DIR}...")
    with zipfile.ZipFile(BUNDLE_ZIP, 'r') as z:
        z.extractall(APF_DIR)
else:
    print(f"[!] {BUNDLE_ZIP} not detected. Please upload s1_smoke_colab_bundle.zip to /content/")

%cd /content/apf

# Fetch and verify pinned S1-mini and Qwen3 tokenizer assets
!python scripts/fetch_tokenizers.py


In [ ]:
# 4. Workspace Configuration
import os, json
WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f"{WORK_DIR}/runs", exist_ok=True)

TRAINER = "/content/apf/research/vf_phase3a/S1_LORA_TRAINER.py"

# Dataset Mode:
# "synthetic" = Stand-in synthetic records (verified offline, zero restricted data)
# "manifest"  = Custom clean dataset manifest file
# "p2b"       = Phase 2B directory (requires formal Hark permission before use)
DATASET_MODE = "synthetic"
MANIFEST_PATH = f"{WORK_DIR}/manifest.json"

In [ ]:
# 5. CPU Preflight Verification
# Validates masking layout, collator, gradient accumulation equivalence, and LoRA wiring
if DATASET_MODE == "synthetic":
    !python $TRAINER verify --synthetic --report /content/work/verify_colab.json --workdir /content/work/tmp
elif DATASET_MODE == "manifest":
    !python $TRAINER verify --manifest $MANIFEST_PATH --report /content/work/verify_colab.json --workdir /content/work/tmp
else:
    !python $TRAINER verify --p2b /content/apf/research/vf_phase2b --report /content/work/verify_colab.json --workdir /content/work/tmp

## Training Approval Gates
All four gates must be filled with real, verified evidence before any GPU training can proceed.
Training will immediately halt if any gate is `false`.

In [ ]:
# 6. Prepare / Inspect gates.json
synth_gates = "/content/apf/research/vf_phase3a/synthetic_fixtures/gates_synthetic_smoke.json"
if DATASET_MODE == "synthetic" and os.path.exists(synth_gates):
    gates = json.load(open(synth_gates))
else:
    gates = json.load(open("/content/apf/research/vf_phase3a/GATES_TEMPLATE.json"))

# Note: The trainer requires explicit user_gpu_approval before GPU execution.
# When authorized by the user for this engineering test:
# gates["user_gpu_approval"] = {"approved": True, "by": "Uzair Kayat", "date": "2026-10-13", "evidence": "Explicitly authorized Colab GPU engineering smoke test"}

GATES_FILE = f"{WORK_DIR}/gates.json"
with open(GATES_FILE, "w") as f:
    json.dump(gates, f, indent=2)
print(json.dumps(gates, indent=2))


## Step 1 — Tiny Trainability Check
Checks model convergence on 16 common + 3 rehearsal rows (8 epochs).
Downloads the authentic `superwhisper/s1-mini` checkpoint (pinned revision `88f6b158`).

In [ ]:
# 7. Run Tiny Profile (Gate-Enforced)
DATA_ARG = "--synthetic" if DATASET_MODE == "synthetic" else f"--manifest {MANIFEST_PATH}"
!python $TRAINER train $DATA_ARG --gates $GATES_FILE --profile tiny --out /content/work/runs/tiny

In [ ]:
# 8. Post-Run Engineering Verification (Metrics, Gradients, Memory & Adapter Reload)
import os, json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

TINY_DIR = "/content/work/runs/tiny"
metrics_file = f"{TINY_DIR}/metrics.jsonl"
assert os.path.exists(metrics_file), f"Metrics file missing: {metrics_file}"
metrics = [json.loads(l) for l in open(metrics_file) if l.strip()]

print(f"[*] Total optimizer updates executed: {len(metrics)}")
first_m, last_m = metrics[0], metrics[-1]
print(f"    Step 1 : loss_common={first_m.get('loss_common')} | grad_norm={first_m.get('grad_norm')}")
print(f"    Step {last_m['step']}: loss_common={last_m.get('loss_common')} | grad_norm={last_m.get('grad_norm')}")

# Assertions
assert first_m.get('loss_common', 0) > 0, "Supervised loss must be non-zero!"
assert first_m.get('grad_norm', 0) > 0, "Gradients must be non-zero!"

# GPU Memory Peak
peak_vram_gb = torch.cuda.max_memory_allocated() / (1024**3)
print(f"[*] Peak GPU VRAM allocated: {peak_vram_gb:.2f} GB")

# Adapter Artifacts
adapter_dir = f"{TINY_DIR}/adapter"
adapter_files = sorted(os.listdir(adapter_dir))
print(f"[*] Saved adapter files: {adapter_files}")
assert "adapter_model.safetensors" in adapter_files, "adapter_model.safetensors missing!"

# Checkpoint Reload Verification
print("[*] Testing adapter checkpoint reload on base model...")
tok = AutoTokenizer.from_pretrained("superwhisper/s1-mini", revision="88f6b15896c73bbb13a3b596e0afe8ea0d5150b4")
base = AutoModelForCausalLM.from_pretrained("superwhisper/s1-mini", revision="88f6b15896c73bbb13a3b596e0afe8ea0d5150b4", torch_dtype=torch.float16).cuda()
reloaded_model = PeftModel.from_pretrained(base, adapter_dir).eval()

probe_ids = tok.encode("probe text <think>", return_tensors="pt").cuda()
with torch.no_grad():
    out = reloaded_model(probe_ids)
assert out.logits is not None
print("[SUCCESS] Adapter reloaded cleanly and executed forward pass on GPU.")


## Step 2 — Controlled Smoke Run
Runs representative subset across all 8 buckets with 15% rehearsal exposure (2 epochs).
Expected run time on T4: ~5-10 minutes.

In [ ]:
# 8. Run Smoke Profile (Gate-Enforced)
!python $TRAINER train $DATA_ARG --gates $GATES_FILE --profile smoke --out /content/work/runs/smoke1

In [ ]:
# 9. Training Metrics Inspection
import pandas as pd
metrics_file = "/content/work/runs/smoke1/metrics.jsonl"
if os.path.exists(metrics_file):
    df = pd.read_json(metrics_file, lines=True)
    display(df[["step", "loss_common", "loss_rehearsal", "grad_norm", "lr", "elapsed_s"]].tail(20))
else:
    print("No training metrics found yet (training has not been executed).")

## Step 3 — Baseline vs Adapter Evaluation
Compares generation between frozen base model and trained LoRA adapter on development examples.
Never tests on the frozen 200-case held-out set.

In [ ]:
# 10. Run Comparison Generation
ADAPTER_DIR = "/content/work/runs/smoke1/adapter"
COMPARE_OUT = "/content/work/runs/smoke1/compare.jsonl"
if os.path.exists(ADAPTER_DIR):
    !python $TRAINER compare $DATA_ARG --adapter $ADAPTER_DIR --out $COMPARE_OUT
else:
    print("No adapter found yet. Train smoke profile first.")

In [ ]:
# 11. (Optional) Checkpoint Backup to Google Drive
# Only if user wishes to persist trained adapter weights to Drive
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/APF_VF1/runs
# !cp -r /content/work/runs/smoke1 /content/drive/MyDrive/APF_VF1/runs/
# !sha256sum /content/work/runs/smoke1/adapter/*